In [0]:
%python
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as F

### Alle Tabellen aus unseren CSV Dateien nehmen und zu Bronze Delta-Tabellen formatieren. 

1. Brands

In [0]:
%python
catalog_name = 'ecommerce'

brand_schema = StructType([
    StructField("brand_code", StringType(), False),
    StructField("brand_name", StringType(), True),
    StructField("category_code", StringType(), True),
    
])

In [0]:
%python
raw_data_path = "/Volumes/ecommerce/source_schema/raw/brands/*.csv"

df = spark.read.option('header', "true").option("delimiter", ",").schema(brand_schema).csv(raw_data_path)
       
df = df.withColumn("_source_file", F.col("_metadata.file_path")) \
    .withColumn("ingested_at", F.current_timestamp())

display(df.limit(5))

brand_code,brand_name,category_code,_source_file,ingested_at
ACME,AcmeTech,CE,dbfs:/Volumes/ecommerce/source_schema/raw/brands/brands.csv,2026-05-14T22:34:26.492Z
NOVW,NovaWave,CE,dbfs:/Volumes/ecommerce/source_schema/raw/brands/brands.csv,2026-05-14T22:34:26.492Z
ZNTH,Zenith,CE,dbfs:/Volumes/ecommerce/source_schema/raw/brands/brands.csv,2026-05-14T22:34:26.492Z
BYTM,ByteMax,CE,dbfs:/Volumes/ecommerce/source_schema/raw/brands/brands.csv,2026-05-14T22:34:26.492Z
ECOT,EcoTone,CE,dbfs:/Volumes/ecommerce/source_schema/raw/brands/brands.csv,2026-05-14T22:34:26.492Z


In [0]:
%python
df.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "True") \
    .saveAsTable(f"{catalog_name}.bronze.brz_brands")

2. Category


In [0]:
%python
category_schema = StructType([
    StructField("category_code", StringType(), False),
    StructField("category_name", StringType(), True)
])
raw_data_path = "/Volumes/ecommerce/source_schema/raw/category/*.csv"
df_category = spark.read.option("header", "true") \
    .option("delimiter", ",") \
    .schema(category_schema) \
    .csv(raw_data_path)

df_category = df_category.withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.col("_metadata.file_path"))

df_category.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_category")

display(df_category.limit(5))

category_code,category_name,_ingested_at,_source_file
ce,Electronics,2026-05-14T23:10:50.420Z,dbfs:/Volumes/ecommerce/source_schema/raw/category/category.csv
app,Apparel,2026-05-14T23:10:50.420Z,dbfs:/Volumes/ecommerce/source_schema/raw/category/category.csv
hnk,Home & Kitchen,2026-05-14T23:10:50.420Z,dbfs:/Volumes/ecommerce/source_schema/raw/category/category.csv
bpc,Beauty & Personal Care,2026-05-14T23:10:50.420Z,dbfs:/Volumes/ecommerce/source_schema/raw/category/category.csv
bks,Books,2026-05-14T23:10:50.420Z,dbfs:/Volumes/ecommerce/source_schema/raw/category/category.csv


3.Produkttabelle

In [0]:
%python
products_schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("sku", StringType(), True),
    StructField("category_code", StringType(), True),
    StructField("brand_code", StringType(), True),
    StructField("color", StringType(), True),
    StructField("size", StringType(), True),
    StructField("material", StringType(), True),
    StructField("weight_grams", StringType(), True),   
    StructField("length_cm", StringType(), True),      
    StructField("width_cm", FloatType(), True),
    StructField("height_cm", FloatType(), True),
    StructField("rating_count", IntegerType(), True),
    StructField("file_name", StringType(), False),
    StructField("ingest_timestamp", TimestampType(), False)
])
raw_data_path = "/Volumes/ecommerce/source_schema/raw/products/*.csv"
df_products = spark.read.option("header", "true") \
    .option("delimiter", ",") \
    .schema(products_schema) \
    .csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

df_products.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_products")

display(df_products.limit(5))

product_id,sku,category_code,brand_code,color,size,material,weight_grams,length_cm,width_cm,height_cm,rating_count,file_name,ingest_timestamp
2000000000015,STCR-HNK-00001,hnk,stcr,White,One-Size,Coton,305g,"22,2",17.1,6.3,0,dbfs:/Volumes/ecommerce/source_schema/raw/products/products.csv,2026-05-14T23:10:58.826Z
2000000000022,HMNS-HNK-00002,hnk,hmns,Silver,One-Size,Steel,682g,"18,2",12.3,3.7,1,dbfs:/Volumes/ecommerce/source_schema/raw/products/products.csv,2026-05-14T23:10:58.826Z
2000000000039,NOVW-CE-00003,ce,novw,Purple,One-Size,Wood,243g,"18,2",13.9,4.2,0,dbfs:/Volumes/ecommerce/source_schema/raw/products/products.csv,2026-05-14T23:10:58.826Z
2000000000046,URTL-APP-00004,app,urtl,Silver,S,Ruber,225g,"17,6",4.6,5.8,50,dbfs:/Volumes/ecommerce/source_schema/raw/products/products.csv,2026-05-14T23:10:58.826Z
2000000000053,GGRN-GRC-00005,grcy,ggrn,Silver,One-Size,Ruber,455g,"27,2",15.8,7.4,-4,dbfs:/Volumes/ecommerce/source_schema/raw/products/products.csv,2026-05-14T23:10:58.826Z


4.Datumtabelle

In [0]:
%python

date_schema = StructType([
    StructField("date", StringType(), True),
    StructField("year", IntegerType(), True),
    StructField("day_name", StringType(), True),
    StructField("quarter", IntegerType(), True),
    StructField("week_of_year", IntegerType(), True)
])

raw_data_path = "/Volumes/ecommerce/source_schema/raw/date/*.csv"

df_date = spark.read.option("header", "true") \
    .option("delimiter", ",") \
    .schema(date_schema) \
    .csv(raw_data_path)

df_date = df_date.withColumn("_ingested_at", F.current_timestamp()) \
    .withColumn("_source_file", F.col("_metadata.file_path"))

df_date.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_date")

display(df_date.limit(5))

date,year,day_name,quarter,week_of_year,_ingested_at,_source_file
01-08-2025,2025,friday,3,-31,2026-05-14T23:23:13.076Z,dbfs:/Volumes/ecommerce/source_schema/raw/date/date.csv
02-08-2025,2025,SATURDAY,3,-31,2026-05-14T23:23:13.076Z,dbfs:/Volumes/ecommerce/source_schema/raw/date/date.csv
03-08-2025,2025,SUNDAY,3,-31,2026-05-14T23:23:13.076Z,dbfs:/Volumes/ecommerce/source_schema/raw/date/date.csv
04-08-2025,2025,MONDAY,3,-32,2026-05-14T23:23:13.076Z,dbfs:/Volumes/ecommerce/source_schema/raw/date/date.csv
05-08-2025,2025,TUESDAY,3,-32,2026-05-14T23:23:13.076Z,dbfs:/Volumes/ecommerce/source_schema/raw/date/date.csv


5. Kundentabelle

In [0]:
%python

customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("phone", StringType(), True),
    StructField("country_code", StringType(), True),
    StructField("country", StringType(), True),
    StructField("state", StringType(), True)
])

raw_data_path = "/Volumes/ecommerce/source_schema/raw/customers/*.csv"

df_customers = spark.read.option("header", "true") \
    .option("delimiter", ",") \
    .schema(customers_schema) \
    .csv(raw_data_path) \
    .withColumn("file_name", F.col("_metadata.file_path")) \
    .withColumn("ingest_timestamp", F.current_timestamp())

df_customers.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(f"{catalog_name}.bronze.brz_customers")

display(df_customers.limit(5))

customer_id,phone,country_code,country,state,file_name,ingest_timestamp
CUST000000000001,917280033536.0,IN,India,MH,dbfs:/Volumes/ecommerce/source_schema/raw/customers/customers.csv,2026-05-14T23:22:52.883Z
CUST000000000002,619489725433.0,AU,Australia,VIC,dbfs:/Volumes/ecommerce/source_schema/raw/customers/customers.csv,2026-05-14T23:22:52.883Z
CUST000000000003,919390066524.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_schema/raw/customers/customers.csv,2026-05-14T23:22:52.883Z
CUST000000000004,917073741793.0,IN,India,TN,dbfs:/Volumes/ecommerce/source_schema/raw/customers/customers.csv,2026-05-14T23:22:52.883Z
CUST000000000005,618478772532.0,AU,Australia,WA,dbfs:/Volumes/ecommerce/source_schema/raw/customers/customers.csv,2026-05-14T23:22:52.883Z
